# Laboratorio didáctico: sistema Sugeno para Iris

Este cuaderno implementa en celdas visibles un clasificador Sugeno de orden cero para Iris. La lógica de pertenencias, reglas, activaciones, promedio ponderado y ponderación por confianza se define aquí y se puede estudiar sin importar esos módulos del proyecto.

**Entradas del sistema:** ancho (`petalwidth`) y largo (`petallength`) del pétalo, en centímetros. **Clases:** setosa = 0, versicolor = 1, virginica = 2.


## 1. Introducción a la lógica difusa y Sugeno

La lógica clásica responde sí/no a una condición. La lógica difusa permite grados entre 0 y 1: una flor puede tener pertenencia 0.4 a “pétalo mediano” y 0.6 a “pétalo grande”. Es útil para representar transiciones graduales sin forzar cortes abruptos.

Una **función de pertenencia** asigna a un valor de entrada un grado \(\mu(x)\in[0,1]\). Una **regla difusa** combina condiciones (“si ancho es pequeño y largo es corto”) y produce un consecuente. En Sugeno de **orden cero**, cada consecuente es una constante numérica. El resultado es el promedio de esas constantes ponderado por la activación de las reglas.

En Mamdani, los consecuentes también son conjuntos difusos y se agregan para luego desfuzzificar. En Sugeno, los consecuentes son constantes (orden cero) o funciones de las entradas (primer orden), de modo que la salida se calcula directamente como promedio ponderado.

Flujo: `entradas → fuzzificación → activaciones → consecuentes → promedio Sugeno → política de clase`.


## 2. Cargar y explorar Iris

Usamos el conjunto Iris de scikit-learn. Para el sistema y sus visualizaciones solo usaremos `petal width (cm)` y `petal length (cm)`. Las otras dos mediciones se omiten del análisis para respetar el modelo definido.

In [ ]:
# Instala solo si falta una dependencia; en Colab suele estar disponible.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
faltantes = [paquete for modulo, paquete in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *faltantes])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from IPython.display import display
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    WIDGETS_DISPONIBLES = True
except ImportError:
    WIDGETS_DISPONIBLES = False

iris = load_iris()
NOMBRES_CLASE = ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
X = pd.DataFrame({'petalwidth': iris.data[:, 3], 'petallength': iris.data[:, 2]})
y = iris.target.astype(int)
datos = X.copy(); datos['clase_id'] = y; datos['clase'] = [NOMBRES_CLASE[i] for i in y]
print('Registros:', len(datos), '| variables del sistema:', list(X.columns))
print('Clases:', dict(zip(NOMBRES_CLASE, np.bincount(y))))
display(datos.head())
display(datos[['petalwidth','petallength']].describe().T)
fig, ax = plt.subplots(figsize=(8,5))
for clase, nombre in enumerate(NOMBRES_CLASE):
    m = y == clase
    ax.scatter(X.loc[m,'petalwidth'], X.loc[m,'petallength'], label=nombre, alpha=.75)
ax.set(xlabel='Petalwidth (cm)', ylabel='Petallength (cm)', title='Iris: entradas usadas por el sistema')
ax.legend(); ax.grid(alpha=.2); plt.show()

La tabla contiene 150 flores, 50 por clase. El gráfico ayuda a ver que estas dos mediciones separan bien a setosa, aunque las otras dos especies tienen zonas de solapamiento. El sistema que construiremos solo recibe estas dos columnas.

## 3. Funciones de pertenencia

Implementamos el trapecio usado por el proyecto. Para parámetros \((a,b,c,d)\), sube linealmente de 0 a 1 en \([a,b]\), permanece en 1 en \([b,c]\), baja hasta 0 en \([c,d]\) y es 0 fuera. Si un extremo coincide (por ejemplo \(c=d\)), forma una rampa que queda en 1 hasta el extremo.

Los trapecios se solapan para suavizar las categorías lingüísticas. **No son los umbrales del árbol de reglas**: para `petalwidth`, las categorías Small/Medium/Large se centran en 0.6 y 1.5, mientras el árbol de decisión contiene 1.7. Se conservan ambos como componentes distintos del modelo.

In [ ]:
def trapezoidal(x, a, b, c, d):
    """Grado trapezoidal, con extremos verticales permitidos (a=b o c=d)."""
    if not a <= b <= c <= d:
        raise ValueError('Se requiere a <= b <= c <= d')
    if x < a or x > d: return 0.0
    if b <= x <= c: return 1.0
    if x < b: return 1.0 if a == b else (x-a)/(b-a)
    return 1.0 if c == d else (d-x)/(d-c)

# Parámetros exactos del proyecto: (a,b,c,d), centímetros.
CONJUNTOS = {
    'petalwidth': {'Small':(-.2,0,.55,.75), 'Medium':(.45,.75,1.4,1.65), 'Large':(1.35,1.65,3,3)},
    'petallength': {'Small':(-1,0,1.8,2.2), 'Medium':(1.8,2.2,4.7,5.1), 'Large':(4.7,5.1,8,8)},
}
def pertenencias(variable, valor):
    return {nombre: trapezoidal(valor,*params) for nombre,params in CONJUNTOS[variable].items()}

def graficar_conjuntos(variable, eje, limites):
    xx=np.linspace(*limites,500)
    for nombre,params in CONJUNTOS[variable].items():
        ax.plot(xx,[trapezoidal(x,*params) for x in xx],label=f'{nombre} {params}')
    for limite in ([.6,1.5,1.7] if variable=='petalwidth' else [2.0,4.9]):
        ax.axvline(limite,color='gray',linestyle='--',alpha=.6)
        ax.text(limite,.05,str(limite),rotation=90,ha='right',fontsize=8)
    ax.set(xlim=limites,ylim=(-.03,1.05),xlabel=f'{variable} (cm)',ylabel='Grado de pertenencia',title=f'Conjuntos lingüísticos: {variable}')
    ax.legend(fontsize=8); ax.grid(alpha=.2)
fig,axes=plt.subplots(1,2,figsize=(14,4))
ax=axes[0]; graficar_conjuntos('petalwidth',ax,(0,2.5))
ax=axes[1]; graficar_conjuntos('petallength',ax,(0,7))
plt.tight_layout(); plt.show()
print('PW=0.60:',pertenencias('petalwidth',.6))
print('PW=1.50:',pertenencias('petalwidth',1.5))
print('PL=4.90:',pertenencias('petallength',4.9))

Por ejemplo, PW=0.60 no tiene por qué ser exclusivamente Small o Medium: los parámetros del trapecio dan pertenencia parcial en las transiciones. En los tramos planos el grado es 1. Los parámetros reproducen literalmente `ConjuntosIris` del proyecto; las líneas verticales también señalan umbrales de reglas para compararlos.

## 4. Fuzzificación

Fuzzificar es evaluar las funciones de pertenencia para cada entrada. Esta operación produce grados lingüísticos, no probabilidades. En el ejemplo `petalwidth=1.6`, `petallength=5.0`, varias categorías pueden tener grado positivo.

In [ ]:
def fuzzificar(entrada):
    return {var: pertenencias(var,float(entrada[var])) for var in ('petalwidth','petallength')}

entrada_ejemplo={'petalwidth':1.6,'petallength':5.0}
grados_ejemplo=fuzzificar(entrada_ejemplo)
tabla_grados=pd.DataFrame(grados_ejemplo).T
display(tabla_grados.round(3))
tabla_grados.plot(kind='bar',figsize=(7,4),ylim=(0,1),title='Grados de pertenencia para PW=1.6, PL=5.0',ylabel='Grado',xlabel='Variable',rot=0)
plt.grid(axis='y',alpha=.2); plt.show()

La altura de cada barra indica cuánto coincide el valor con una categoría lingüística. Una entrada puede activar parcialmente categorías vecinas; esa es la transición difusa, no una afirmación de que la flor tenga una probabilidad de pertenecer a ellas.

## 5. Las cinco reglas del árbol

Los umbrales de las reglas se convierten en rampas suaves de ancho 0.2 cm, tal como hace el proyecto. Definimos \(\mu_{>t}(x)\) como 0 bajo \(t-0.1\), crece linealmente y llega a 1 sobre \(t+0.1\); \(\mu_{\le t}=1-\mu_{>t}\). El operador AND será mínimo.

La categoría lingüística `petalwidth Medium` usa el trapecio hasta 1.5, pero las reglas R3–R5 usan el límite de árbol 1.7. No sustituimos 1.7 por 1.5 ni usamos los grados lingüísticos como si fueran esos predicados: son componentes diferentes definidos por el ejercicio.

In [ ]:
def rampa_mayor(x, limite, ancho=.2):
    inicio,fin=limite-ancho/2,limite+ancho/2
    if x <= inicio+1e-12: return 0.0
    if x >= fin-1e-12: return 1.0
    return (x-inicio)/ancho
def rampa_menor_igual(x, limite, ancho=.2):
    return 1-rampa_mayor(x,limite,ancho)

REGLAS=[
 {'id':'R1','texto':'PW <= 0.6 → Setosa','logica':'μ≤0.6(PW)','clase':0,'consecuente':0.0},
 {'id':'R2','texto':'PW > 1.7 → Virginica','logica':'μ>0.6(PW) AND μ>1.7(PW)','clase':2,'consecuente':2.0},
 {'id':'R3','texto':'0.6 < PW <= 1.7 y PL <= 4.9 → Versicolor','logica':'μ>0.6(PW) AND μ≤1.7(PW) AND μ≤4.9(PL)','clase':1,'consecuente':1.0},
 {'id':'R4','texto':'0.6 < PW <= 1.7, PL > 4.9 y PW <= 1.5 → Virginica','logica':'μ>0.6(PW) AND μ≤1.7(PW) AND μ>4.9(PL) AND μ≤1.5(PW)','clase':2,'consecuente':2.0},
 {'id':'R5','texto':'0.6 < PW <= 1.7, PL > 4.9 y PW > 1.5 → Versicolor','logica':'μ>0.6(PW) AND μ≤1.7(PW) AND μ>4.9(PL) AND μ>1.5(PW)','clase':1,'consecuente':1.0},
]
display(pd.DataFrame([{k:r[k] for k in ('id','texto','logica','consecuente')} for r in REGLAS]))

R1 inspects only width; R2 expresses the high-width branch; R3 handles the moderate-width, shorter-length region; R4 and R5 split the long-length region at PW=1.5. Each rule has a constant class consequent for zero-order Sugeno.

## 6. Inferencia y activación

La **pertenencia** describe una entrada en una categoría; la **activación** mide cuánto se satisface una regla completa; el **consecuente** es el valor de salida asociado a esa regla. Con AND mínimo, la activación es el mínimo de los grados de sus condiciones.

In [ ]:
def activaciones_reglas(pw,pl):
    g06=rampa_mayor(pw,.6); l06=1-g06
    g17=rampa_mayor(pw,1.7); l17=1-g17
    l49=rampa_menor_igual(pl,4.9); g49=1-l49
    l15=rampa_menor_igual(pw,1.5); g15=1-l15
    return {'R1':l06,'R2':min(g06,g17),'R3':min(g06,l17,l49),
            'R4':min(g06,l17,g49,l15),'R5':min(g06,l17,g49,g15)}

def explicar_activaciones(pw,pl):
    act=activaciones_reglas(pw,pl)
    return pd.DataFrame([{'Regla':r['id'],'Activación':act[r['id']], 'Consecuente':r['consecuente'], 'Clase':NOMBRES_CLASE[r['clase']]} for r in REGLAS])

for ejemplo in [(0.4,1.0),(1.0,3.0),(1.6,5.0)]:
    print('PW, PL =',ejemplo); display(explicar_activaciones(*ejemplo).round(2))

Los ejemplos corresponden a regiones distintas del árbol: baja anchura, zona media y longitud grande. Las rampas pueden hacer que reglas vecinas compartan activación cerca de límites; no se interpreta eso como un cambio en las condiciones originales. El gráfico siguiente resume las activaciones del ejemplo del ejercicio.

In [ ]:
activ_ejemplo=activaciones_reglas(1.6,5.0)
plt.figure(figsize=(7,4)); plt.bar(list(activ_ejemplo),list(activ_ejemplo.values()),color='#527da8')
plt.ylim(0,1.05); plt.ylabel('Activación'); plt.title('Activación de reglas para PW=1.6, PL=5.0'); plt.grid(axis='y',alpha=.2); plt.show()

## 7. Motor Sugeno de orden cero

Para activaciones \(w_i\) y consecuentes constantes \(z_i\), Sugeno calcula \(y=rac{\sum_i w_i z_i}{\sum_i w_i}\). El numerador suma contribuciones ponderadas; el denominador normaliza para obtener un promedio. Si la suma de activaciones es cero, no hay inferencia y devolvemos `None` explícitamente.

In [ ]:
def promedio_ponderado(pesos, consecuentes):
    denominador=sum(pesos.values())
    if denominador==0: return None
    return sum(pesos[r]*consecuentes[r] for r in pesos)/denominador

def inferir(pw,pl,confianzas=None):
    activ=activaciones_reglas(pw,pl)
    consecuentes={r['id']:r['consecuente'] for r in REGLAS}
    pesos={r:activ[r]*(confianzas[r] if confianzas is not None else 1.0) for r in activ}
    salida=promedio_ponderado(pesos,consecuentes)
    return {'grados':fuzzificar({'petalwidth':pw,'petallength':pl}), 'activaciones':pesos,
            'consecuentes':consecuentes,'productos':{r:pesos[r]*consecuentes[r] for r in pesos},
            'salida':salida}

salida_demo=inferir(1.6,5.0)
display(pd.DataFrame({'w':salida_demo['activaciones'],'z':salida_demo['consecuentes'],'w*z':salida_demo['productos']}))
num=sum(salida_demo['productos'].values()); den=sum(salida_demo['activaciones'].values())
print(f'y = {num:.6f} / {den:.6f} = {salida_demo["salida"]:.6f}')
assert promedio_ponderado({'R1':0.0},{'R1':0.0}) is None

La tabla deja visible cada producto \(w_i z_i\). La división final da una salida continua. La última aserción verifica explícitamente el caso sin activaciones.

## 8. Convertir la salida a una clase

Una salida Sugeno es real, pero la etiqueta Iris es discreta. Usamos como política uniforme la clase entera más cercana, con empates hacia el índice menor: 0 para \(y\le0.5\), 1 para \(0.5<y\le1.5\), 2 para \(y>1.5\), recortando al rango 0..2. En las fronteras exactas gana la clase de menor índice.

El redondeo es simple, pero trata las clases como ordenadas y con distancias iguales. Los intervalos hacen explícitos esos límites. No hay garantía de que esta política sea óptima para toda aplicación; aquí la dejamos fijada para que las dos variantes se evalúen justamente.

In [ ]:
def clase_desde_salida(y):
    if y is None: return None
    # Empates entre clases vecinas: gana el índice menor.
    return max(0,min(2,int(y+.5-1e-12)))
for y_demo in [0.2,.5,.8,1.5,1.8]:
    print(f'y={y_demo:.1f} → clase {clase_desde_salida(y_demo)} ({NOMBRES_CLASE[clase_desde_salida(y_demo)]})')

## 9. Ejemplo interactivo

Los sliders recalculan pertenencias, activaciones, consecuentes, productos, salida y clase. Si los widgets no están disponibles, edita `pw_manual` y `pl_manual` en la celda siguiente y ejecútala.

In [ ]:
def mostrar_entrada(pw,pl):
    r=inferir(float(pw),float(pl)); yhat=clase_desde_salida(r['salida'])
    print(f'Entrada PW={pw:.2f} cm, PL={pl:.2f} cm')
    print('Grados de pertenencia (no probabilidades):'); display(pd.DataFrame(r['grados']).T.round(3))
    print('Activaciones, consecuentes y productos:'); display(pd.DataFrame({'activación':r['activaciones'],'consecuente':r['consecuentes'],'producto':r['productos']}))
    print('Salida Sugeno:',r['salida'],'→ clase:',NOMBRES_CLASE[yhat] if yhat is not None else 'sin regla activa')
    fig,ax=plt.subplots(figsize=(7,3)); ax.bar(list(r['activaciones']),list(r['activaciones'].values()),color='#6b8e75')
    ax.set(ylim=(0,1.05),ylabel='Activación',title='Reglas activadas'); ax.grid(axis='y',alpha=.2); plt.show()
    activas=[k for k,v in r['activaciones'].items() if v>0]
    print('Lectura: las reglas con activación positiva contribuyen a la media; su consecuente indica la clase sugerida.')

if WIDGETS_DISPONIBLES:
    salida_widget=widgets.interactive_output(mostrar_entrada,{
      'pw':widgets.FloatSlider(value=1.6,min=0,max=2.5,step=.01,description='Petalwidth',continuous_update=False),
      'pl':widgets.FloatSlider(value=5.0,min=0,max=7,step=.01,description='Petallength',continuous_update=False)})
    display(salida_widget)
else:
    print('ipywidgets no disponible; usa los valores editables debajo.')
pw_manual,pl_manual=1.6,5.0 # editar estos valores si no hay widgets
mostrar_entrada(pw_manual,pl_manual)

## 10. Sugeno ponderado por confianza de las hojas

Las confianzas documentadas son proporciones de aciertos en hojas del árbol: R1 50/50, R2 46/47, R3 48/49, R4 3/3 y R5 3/4. No necesariamente son probabilidades calibradas. En esta variante, \(w_i'=w_i c_i\), y luego se calcula la misma media ponderada con \(w_i'\). Es una variante del modelo, no el Sugeno convencional.

In [ ]:
DATOS_HOJAS={'R1':(50,0),'R2':(46,1),'R3':(48,1),'R4':(3,0),'R5':(3,1)}
CONFIANZAS={r:a/(a+b) for r,(a,b) in DATOS_HOJAS.items()}
tabla_conf=pd.DataFrame([{'Regla':r,'Correctos':a,'Incorrectos':b,'Confianza':CONFIANZAS[r]} for r,(a,b) in DATOS_HOJAS.items()])
display(tabla_conf.assign(Confianza=tabla_conf['Confianza'].map(lambda x:f'{x:.2%}'))) 
conv=inferir(1.6,5.0); ponderado=inferir(1.6,5.0,CONFIANZAS)
comparacion_pesos=pd.DataFrame({'Activación normal':conv['activaciones'],'Confianza':CONFIANZAS,'Activación ponderada':ponderado['activaciones'],'Consecuente':{r['id']:r['consecuente'] for r in REGLAS}})
display(comparacion_pesos.round(3))
comparacion_pesos[['Activación normal','Activación ponderada']].plot(kind='bar',figsize=(8,4),title='Impacto de ponderar por confianza',ylabel='Peso de regla',rot=0)
plt.grid(axis='y',alpha=.2); plt.show()
print('Salida convencional:',conv['salida'],'clase',clase_desde_salida(conv['salida']))
print('Salida ponderada:',ponderado['salida'],'clase',clase_desde_salida(ponderado['salida']))

La multiplicación reduce el peso relativo de reglas menos confiables. La confianza de R5 es 75%, así que una activación R5=0.8 pasa a peso 0.6. Las reglas R1 y R4 conservan su peso porque las muestras de sus hojas no tuvieron errores.

## 11. Evaluación sobre Iris

Evaluamos las 150 observaciones, exactamente las mismas para ambas variantes, usando las dos entradas y la política de clase definida arriba. Informamos métricas y matriz de confusión sin ajustar reglas ni esconder errores.

In [ ]:
def predecir_conjunto(X, confianzas=None):
    pred=[]
    for pw,pl in X[['petalwidth','petallength']].itertuples(index=False,name=None):
        r=inferir(pw,pl,confianzas); clase=clase_desde_salida(r['salida'])
        pred.append(-1 if clase is None else clase)
    return np.asarray(pred,dtype=int)

pred_conv=predecir_conjunto(X)
pred_conf=predecir_conjunto(X,CONFIANZAS)
assert len(pred_conv)==len(pred_conf)==len(y) and np.array_equal(y,y.copy())
def mostrar_metricas(nombre,pred):
    print(nombre,'accuracy:',round(accuracy_score(y,pred),4))
    print(classification_report(y,pred,labels=[0,1,2],target_names=NOMBRES_CLASE,zero_division=0,digits=3))
    return confusion_matrix(y,pred,labels=[0,1,2])
cm_conv=mostrar_metricas('Sugeno convencional',pred_conv)
cm_conf=mostrar_metricas('Sugeno ponderado por confianza',pred_conf)
fig,axs=plt.subplots(1,2,figsize=(10,4))
for ax,cm,titulo in zip(axs,[cm_conv,cm_conf],['Convencional','Ponderado por confianza']):
    ax.imshow(cm,cmap='Blues'); ax.set(title=titulo,xlabel='Predicción',ylabel='Clase real',xticks=range(3),yticks=range(3),xticklabels=['Setosa','Versicolor','Virginica'],yticklabels=['Setosa','Versicolor','Virginica'])
    for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
plt.tight_layout(); plt.show()

Las filas de la matriz son clases reales y las columnas predichas. Los valores de diagonal son aciertos; los de fuera de diagonal muestran confusiones. La ponderación cambia el peso de reglas, no las muestras ni la política de conversión.

## 12. Visualizar el comportamiento en el plano

El fondo representa la predicción de la regla difusa en cada punto del plano; los círculos muestran etiquetas reales y las cruces errores de la predicción en los datos. Paneles separados muestran clases reales y predichas; el tercero destaca errores y regiones. El contorno indica la regla con mayor activación (prioridad al primer orden en empates).

In [ ]:
pw_grid=np.linspace(0,2.5,140); pl_grid=np.linspace(0,7,180)
ZZ=np.zeros((len(pl_grid),len(pw_grid)),dtype=int); RR=np.zeros_like(ZZ)
for i,pl in enumerate(pl_grid):
    for j,pw in enumerate(pw_grid):
        out=inferir(pw,pl); ZZ[i,j]=clase_desde_salida(out['salida'])
        RR[i,j]=max(range(5),key=lambda k:out['activaciones'][f'R{k+1}'])
errores=pred_conv!=y
pred_color=['#4c78a8','#f2a541','#59a14f']
fig,axs=plt.subplots(1,3,figsize=(18,5))
for ax,title,labels in [(axs[0],'Clase real',y),(axs[1],'Clase predicha',pred_conv)]:
    for k,nombre in enumerate(NOMBRES_CLASE):
        m=labels==k
        ax.scatter(X.loc[m,'petalwidth'],X.loc[m,'petallength'],s=25,label=nombre,color=pred_color[k],edgecolors='black',linewidths=.25)
    ax.set(title=title,xlabel='Petalwidth (cm)',ylabel='Petallength (cm)'); ax.legend(fontsize=8)
axs[2].contourf(pw_grid,pl_grid,ZZ,levels=[-.5,.5,1.5,2.5],colors=pred_color,alpha=.22)
axs[2].contour(pw_grid,pl_grid,RR,levels=np.arange(-.5,5,.5),colors='gray',linewidths=.7)
axs[2].scatter(X['petalwidth'],X['petallength'],c=[pred_color[k] for k in y],s=18,alpha=.65,label='Muestras Iris')
axs[2].scatter(X.loc[errores,'petalwidth'],X.loc[errores,'petallength'],marker='x',s=70,color='red',label='Error')
axs[2].set(title='Regiones predichas y reglas dominantes',xlabel='Petalwidth (cm)',ylabel='Petallength (cm)'); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Errores en datos:',int(errores.sum()),'de',len(y))


El fondo proviene de evaluar el motor real sobre una malla; las marcas provienen de Iris y de las predicciones calculadas sobre las mismas observaciones. Las fronteras de regla dominante solo indican qué regla tiene el peso mayor, no una clase de certeza absoluta.

## 13. Incertidumbre e interpretación

- Varias reglas pueden activarse a la vez por las rampas suaves. La salida combina sus consecuentes según el peso.
- Activación baja significa que una condición difusa se satisface poco; no es probabilidad de error.
- Una confianza menor reduce la influencia en la variante ponderada, pero la confianza de hoja tampoco es una probabilidad calibrada.
- Si todas las activaciones son cero, el denominador es cero y la inferencia devuelve `None`. En el conjunto de reglas actual las regiones cubren las entradas habituales.
- Dos entradas cercanas pueden cambiar los pesos gradualmente; cerca de una frontera también puede cambiar la clase discreta al cruzar 0.5 o 1.5.

Pertenencia, activación y confianza son conceptos distintos: grado de coincidencia con conjunto; fuerza de una regla completa; proporción histórica de aciertos de su hoja, respectivamente.

In [ ]:
# Busca ejemplos reales próximos en sus entradas y compara pertenencias, activaciones y salidas.
orden=X.sort_values('petalwidth').index.to_list()
par=None
for a,b in zip(orden,orden[1:]):
    distancia=np.linalg.norm(X.loc[a].to_numpy()-X.loc[b].to_numpy())
    if distancia<.035 and y[a]!=y[b]: par=(a,b); break
if par:
    for idx in par:
        row=X.loc[idx]; r=inferir(row.petalwidth,row.petallength)
        print(f"índice={idx}, real={NOMBRES_CLASE[y[idx]]}, PW={row.petalwidth:.2f}, PL={row.petallength:.2f}, salida={r['salida']:.3f}, predicha={NOMBRES_CLASE[clase_desde_salida(r['salida'])]}")
else:
    print('No se encontró par de clases distintas a distancia menor que 0.035 cm en esta búsqueda.')

## 14. Conclusiones y preparación para el examen

Sugeno fuzzifica entradas, calcula activaciones, asigna consecuentes y obtiene una media ponderada. En orden cero, los consecuentes son constantes; en primer orden, cada consecuente puede ser una función lineal de las entradas. Mamdani usa conjuntos difusos de salida y requiere agregación/desfuzzificación; Sugeno produce directamente una salida numérica.

La salida se convierte aquí a clase por vecino numérico más cercano, con empates hacia la clase menor. El promedio ponderado es útil porque combina reglas simultáneamente activas en una salida continua. Si no hay activaciones, no se define salida.

### Preguntas y respuestas razonadas

1. **¿Qué es un grado de pertenencia?** Un valor \([0,1]\) de compatibilidad de una entrada con un conjunto lingüístico; no es una probabilidad.
2. **¿Cómo se calcula una activación?** Se combinan grados de las condiciones de la regla; aquí AND usa el mínimo.
3. **¿Qué representa el consecuente?** En orden cero es una constante de salida asociada con una regla (0, 1 o 2 aquí).
4. **¿Por qué se usa un promedio ponderado?** Las reglas activas contribuyen proporcionalmente a su activación y el denominador normaliza los pesos.
5. **¿Qué ocurre si todas las activaciones son cero?** El denominador es cero, no existe salida inferida y devolvemos `None`.
6. **¿Diferencia entre regla activada y confiable?** Activación depende de la entrada actual; confianza resume aciertos observados en la hoja de entrenamiento.
7. **¿Cómo pasar de salida a clase?** Aplicamos una política declarada: clase numérica más cercana, con empates al índice menor.
8. **¿Orden cero vs primer orden?** Constantes por regla frente a consecuentes lineales/afines en las entradas.

### Comprobaciones del ejercicio

Se mantienen los trapecios y las rampas del proyecto, los cinco umbrales y consecuentes, y las cinco confianzas indicadas. El umbral lingüístico 1.5 y el umbral del árbol 1.7 se explican por separado.